# 04 · Validate — geometry preservation, method comparison, docking & MD

**Standard slot:** *validate (in silico).* **For Project 18 this is the benchmark:** the
**catalytic-geometry preservation rate**, a **scaffolding-method comparison** (RFdiffusion2 vs
Riff-Diff vs motif scaffolding) `[extension]`, and the docking / active-site-MD figures (D3 pt2).

Needs `results/campaign.csv` (+ `results/ranked.csv` from notebook 03).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Catalytic-geometry preservation — the headline figure
Distribution of catalytic-geometry RMSD vs the 0.5 Å pass bar. The fraction left of the line is the
**preservation rate** — the metric that most distinguishes scaffolding methods. (Numbers here are
SYNTHETIC mock values; on Colab they come from real AF2 predictions.)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

camp = pd.read_csv("results/campaign.csv")
cut = 0.5

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.hist(camp["catalytic_geom_rmsd"], bins=20)
ax.axvline(cut, color="k", ls="--", lw=1, label=f"pass < {cut} A")
ax.set_xlabel("catalytic-geometry RMSD vs theozyme (A)  [SYNTHETIC]")
ax.set_ylabel("designs"); ax.set_title("Catalytic-geometry preservation")
ax.legend(); plt.tight_layout()
plt.savefig("results/catalytic_geometry_hist.png", dpi=150); plt.show()

rate = 100 * (camp["catalytic_geom_rmsd"] <= cut).mean()
print(f"overall catalytic-geometry preservation rate = {rate:.1f}%  [SYNTHETIC demo]")

## 2 · Scaffolding-method comparison `[extension]`
Compare the preservation rate (and hit rate) across scaffolding methods. In the real campaign these
are RFdiffusion2 vs Riff-Diff vs classic RFdiffusion motif scaffolding; here a single mock method is
present, so this cell shows the *shape* of the comparison you will populate on Colab.

In [ ]:
by_method = (camp.assign(pass_geom=camp["catalytic_geom_rmsd"] <= 0.5)
                 .groupby("scaffold_method")
                 .agg(n=("design_id", "size"),
                      geom_pass_rate=("pass_geom", "mean"),
                      mean_plddt_cat=("plddt_catalytic", "mean"))
                 .reset_index())
by_method["geom_pass_rate"] = (100 * by_method["geom_pass_rate"]).round(1)
print("Scaffolding-method comparison (populate with real methods on Colab):")
print(by_method.to_string(index=False))
print("\n[SYNTHETIC] On Colab: compare rfdiffusion2 vs riffdiff vs rfdiffusion on the SAME theozyme.")

## 3 · Substrate docking + active-site MD (top candidates)
Docking (AutoDock Vina) checks the substrate **fits and is oriented** toward the catalytic base — not
affinity, not activity. Short MD (OpenMM) checks the active site doesn't drift. Plot the two for the
ranked survivors as orthogonal evidence.

In [ ]:
import matplotlib.pyplot as plt
# ranked.csv comes from the shared filter (fp.Design fields); vina_score lives in campaign.csv,
# so merge it back by design_id for the docking-vs-MD view.
try:
    ranked = pd.read_csv("results/ranked.csv")
    ranked = ranked.merge(camp[["design_id", "vina_score"]], on="design_id", how="left")
except FileNotFoundError:
    ranked = camp.copy()

top = ranked.head(min(20, len(ranked)))
fig, ax = plt.subplots(figsize=(5.2, 3.4))
sc = ax.scatter(top["vina_score"], top["md_rmsd"],
                c=top["catalytic_geom_rmsd"], cmap="viridis")
ax.set_xlabel("Vina substrate-fit score (more negative = better fit)  [SYNTHETIC]")
ax.set_ylabel("active-site MD RMSD (A)  [SYNTHETIC]")
ax.set_title("Top candidates: substrate fit vs active-site stability")
fig.colorbar(sc, label="catalytic-geom RMSD (A)")
plt.tight_layout(); plt.savefig("results/docking_md.png", dpi=150); plt.show()
print("Lower-left + dark points (good fit, stable, good geometry) are the best candidates [SYNTHETIC].")

## 4 · Honest hit-rate accounting
Report N(pass all layers) / N(generated), and remind the reader of the field reality: even a good
preservation rate is **not** an activity rate. Geometry ≠ catalysis; a kinetic assay is required.

In [ ]:
n_total = len(camp)
try:
    ranked = pd.read_csv("results/ranked.csv")
    n_hits = int((ranked["layers_passed"] >= 3).sum())
except Exception:
    n_hits = int((camp["catalytic_geom_rmsd"] <= 0.5).sum())
print(f"Hit-rate accounting [SYNTHETIC demo]:")
print(f"  generated            : {n_total}")
print(f"  pass all filter layers: {n_hits}  ({100*n_hits/max(n_total,1):.1f}%)")
print("\nREALITY CHECK: de novo enzyme activity rates are <1% without directed evolution, and")
print("in-silico catalytic geometry does NOT guarantee activity. Only a kinetic assay decides.")

## D3 (part 2) checklist
- [ ] Catalytic-geometry preservation histogram (`results/catalytic_geometry_hist.png`) + rate.
- [ ] Scaffolding-method comparison table/figure (real methods on Colab) `[extension]`.
- [ ] Docking + active-site-MD figure on the ranked top set.
- [ ] Honest hit-rate accounting with the "geometry ≠ activity" caveat stated.

**Next:** `05_validation_plan.ipynb` — the kinetic-assay plan + controls + directed-evolution stretch.